# Select the correct environment and import necessary modules

Select the correct kernel.

I recommend creating a new environment. Then to select it :
- Select Kernel
    - Select another kernel
        - Jupyter kernel
            - Smart Resume an Job matcher (or any kernel name)

In [ ]:
import sys# To find agents python files

In [2]:
print(sys.executable)
# Should see something like : ...\Smart_Resume_to_Job_Matcher\.venv\Scripts\python.exe

c:\Users\Sebastien\Desktop\LLM and GenAI\Smart_Resume_to_Job_Matcher\.venv\Scripts\python.exe


In [ ]:
import importlib# To reload modified python files for agents

In [2]:
from pathlib import Path

# Pull models and embeddings

In a first terminal window, launch the ollama app:
~~~
ollama serve
~~~

Then pull the model and embedding from another terminal window:
~~~
ollama pull llama3
ollama pull nomic-embed-text
ollama pull mxbai-embed-large
~~~

To eventually delete all models, use:
~~~
ollama list
ollama rm <model-name>
~~~

# Resume and Job description loader

~~~
project/
├── data/
│ ├── resumes/
| | └── resumes.csv # already partially preprocessed
│ └── jobs/
| | └── job_postings.csv # filtered with only the real jobs applications
~~~

In this part, we need to transform each resume and job description into one single text.

In [13]:
# Add the parent folder of ingestion to sys.path
project_root = Path("..").resolve()  # notebooks/ is one level down
sys.path.append(str(project_root))

In [9]:
# This code is to reload the modified python files for the agent
import ingestion.preprocess
# Reload the file to take into account the changes
importlib.reload(ingestion.preprocess)

<module 'ingestion.preprocess' from 'C:\\Users\\Sebastien\\Desktop\\LLM and GenAI\\Smart_Resume_to_Job_Matcher\\ingestion\\preprocess.py'>

In [6]:
from ingestion.preprocess import clean_text, resumes_to_raw_text, jobs_to_raw_text
# Load resumes and job only reads the csv file and transform each line into a json like format

In [7]:
# Now, we get the resumes and jobs as lists of formatted JSON files, 
#   with the id and formatted text for each file
resumes = resumes_to_raw_text("../data/resumes/resumes.csv")
jobs = jobs_to_raw_text("../data/jobs/job_postings.csv")

In [60]:
print("Loaded", len(resumes), "resumes")
print("Loaded", len(jobs), "job descriptions")

print("Sample resume:", resumes[482])
print("Sample job:", jobs[15489])

Loaded 13389 resumes
Loaded 17014 job descriptions
Sample resume: {'resume_id': 483, 'text': 'Category: Aviation. Resume text: qualifications stacy sample 702 8000000 cell 0000emailcom flight ratings faa atpairline transport pilot cfirotorcrafthelicopter s70 s92 typeratings medical flight time total 6867 pilot command 5673 night 1342 pilot 1194 night vision goggle 528 helicopter multiengine 4660 offshoreoverwater 574 faa class 1 cross country 3847 external load 263 instrument ifrhood 832 ems 541 actual weather 369 mountain 2683 experience sikorsky s76 s92 test pilot jul 2007present bell 430 corporate ifr captain jan 2006jun 2007 sikorsky s76 ems captain oct 2003dec 2005 sikorsky aircraft corporation coatesville pa s76 592 production test pilotmaintenance test pilot engineering test flight civil foreign military flight instructor customer demonstrations international ifr vfr overwater experience elite aviationlos angeles ca single pilot bell 430 corporate ifr captainoffairport landing l

# Normalize text

In [111]:
import agents.normalization_agent
# Reload the file to take into account the changes
importlib.reload(agents.normalization_agent)

<module 'agents.normalization_agent' from 'C:\\Users\\Sebastien\\Desktop\\LLM and GenAI\\Smart_Resume_to_Job_Matcher\\agents\\normalization_agent.py'>

In [ ]:
from agents.normalization_agent import normalize_resume, normalize_job

### Example

In [24]:
# Now, we try it on the first resume and job text
resume_text = resumes[0]["text"]
job_text = jobs[0]["text"]
print(resume_text)
print(job_text)

Category: Accountant. Resume text: education omba executive leadership university texas 20162018 bachelor science accounting richland college 20052008 training certifications certified management accountant cma certified financial modeling valuation analyst compliance antimoney laundering 092016 american institute banking certified public account cpa lean six sigma green belt certified trade products financial regulations 082016 american institute banking achievements speaker bringing leader within 082019 successfully presented empowering speech leadership 500 participants speaker dallas convention cpas 032019 successfully delivered seminar 3k cpas convention guests teaching experience online teacher udemy 2017 taught online accounting nonaccountant course udemy similar online teaching platforms developed effective teaching modules materials curriculum target students took feedbacks students assist improving teaching methodology materials professional memberships affiliations american 

In [25]:
# The llama model is initialized inside the agent (so we only need to run the agent)
normalized_resume = normalize_resume(resume_text)
normalized_job = normalize_job(job_text)

print(normalized_resume)
print(normalized_job)

{'skills': ['Microsoft Office', 'QuickBooks', 'ERP', 'SAP', 'Oracle', 'Hyperion', 'SQL'], 'experience': ['Accountant – Omba Executive Leadership (2016-2018): Managed financial regulations and compliance.', 'Speaker, Dallas Convention CPAs (2019): Successfully presented empowering speech to 500 participants.'], 'education': ['Bachelor of Science in Accounting, Richland College (2005-2008)', 'Executive Leadership, University of Texas (2016-2018)'], 'certifications': ['CMA', 'CFMVA', 'CPA', 'Lean Six Sigma Green Belt'], 'industries': ['Financial Regulations']}
{'job_title': 'Marketing Intern', 'required_skills': ['Content management systems', 'Food52 editorial voice and aesthetic', 'Social media (Twitter, Facebook, Pinterest)', 'Problem-solving', 'Collaboration'], 'required_experience': 'Internship', 'required_education': 'Not specified', 'industry': 'Media'}


/!\ INFO /!\
 - RESUMES

Preprocess gives in a text the whole resume text

For normalized resumes, should give the infos :
{'skills': [], 'experience': [], 'education': [], 'certifications': [], 'industries': []}


 - JOB DESCRIPTIONS

Preprocess gives in a text : title, location, description, requirements, employment_type, required_experience, required_education, industry, function

For normalized job, should give the following infos : 
{'job_title': '...', 'required_skills': [], 'required_experience': '', 'required_education': '', 'industry': ''}

### Complete normalization

Now that we have an example with one resume and one job, we need to do it for all resumes and jobs.

In this step, I decided to put normalized resumes and jobs into two distinct lists, each containing the normalized representation of resumes and job descriptions. The objects will look like this :
```python 
normalized_resumes = [
    {'resume_id': 1, 
    'norm_text': {'skills': [...], 'experience': [...], 'education': [...], 'certifications': [...], 'industries': [...]}
    },
    ...
]

normalized_jobs = [
    {'job_id': 1, 
    'job_description': {'job_title': '...', 'required_skills': [...], 'required_experience': '...', 'required_education': '...', 'industry': '...'}
    },
]
```

However, since the quantity of resumes and job description is very high, the computation takes time. In case it crashes in the middle, I created a checkpoint file to save progress every 10 normalized elements.

/!\ 
**Léger problème** : les skills ne semblent pas normalizés. J'ai pu voir "MS Office" mais aussi "Microsoft Excel", "Microsoft Word" et "Microsoft Office", alors que j'ai explicitement demander la normalization dans le prompt pour le llm.

-> Problème réglé grâce à une "normalization forcée".

#### Resumes (13389 elements) **DONE**

In [15]:
import os
import json
from tqdm import tqdm
from pathlib import Path

In [16]:
# We first prepare checkpoints (very long computation)
## Checkpoint directory to save partial results
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

## File to save intermediate results
resume_checkpoint = checkpoint_dir / "normalized_resumes.json"

## Load existing checkpoint if available
if resume_checkpoint.exists():
    print("Checkpoint found")
    with open(resume_checkpoint, "r") as f:
        normalized_resumes = json.load(f)
    processed_ids = {r['resume_id'] for r in normalized_resumes}
    print(len(processed_ids), "resumes")
else:
    normalized_resumes = []
    processed_ids = set()

Checkpoint found
12020 resumes


In [17]:
# Then we loop through resumes, adding a progress bar 
for resume in tqdm(resumes, desc="Normalizing resumes"):
    resume_id = resume['resume_id']
    # Skip resumes already processed
    if resume_id in processed_ids:
        continue
    resume_text = resume['text']
    # We normalize the text
    norm_resume_text = normalize_resume(resume_text)# --> None if normalization failed after 3 attempts <--
    # We join everything in a dictionnary for clarity
    normalized_resume = {
        'resume_id': resume_id,
        'norm_text': norm_resume_text
    }
    # We put the result in a list
    normalized_resumes.append(normalized_resume)
    processed_ids.add(resume_id)
    # We save a checkpoint (every 10 resumes)
    if len(normalized_resumes) % 10 == 0:
        with open(resume_checkpoint, "w") as f:
            json.dump(normalized_resumes, f, indent=2)

# Final save
with open(resume_checkpoint, "w") as f:
    json.dump(normalized_resumes, f, indent=2)

Normalizing resumes:  90%|████████▉ | 12030/13389 [00:45<00:10, 132.53it/s] 

⚠ Attempt 1 failed for resume. Retrying...


Normalizing resumes:  91%|█████████ | 12120/13389 [08:13<1:48:26,  5.13s/it]

⚠ Attempt 1 failed for resume. Retrying...


Normalizing resumes:  91%|█████████ | 12209/13389 [16:02<1:51:22,  5.66s/it]

...

Normalizing resumes: 100%|█████████▉| 13354/13389 [1:57:39<02:16,  3.91s/it]

⚠ Attempt 1 failed for resume. Retrying...


Normalizing resumes: 100%|█████████▉| 13360/13389 [1:58:21<02:54,  6.01s/it]

⚠ Attempt 1 failed for resume. Retrying...


Normalizing resumes: 100%|██████████| 13389/13389 [2:00:52<00:00,  1.85it/s]


The failed resume are still put into the JSON. I have to take care of them.

In [18]:
# Load normalized resumes
with open(resume_checkpoint, "r") as f:
    normalized_resumes = json.load(f)
# We drop all the elements in normalized_resumes with 'norm_text'=None
correctly_normalized_resumes = [r for r in normalized_resumes if r['norm_text'] is not None]
# Let's see how much resumes where not correctly normalized
print("Number of resumes normalized :", len(correctly_normalized_resumes), "/", len(normalized_resumes))
print("Equivalent in %:", 100*len(correctly_normalized_resumes)/len(normalized_resumes), "%")

Number of resumes normalized : 13314 / 13389
Equivalent in %: 99.43983867353798 %


 - Number of resumes normalized : 13314 / 13389
 - Equivalent in %: 99.43983867353798 %

In [19]:
# Resave
with open(resume_checkpoint, "w") as f:
    json.dump(correctly_normalized_resumes, f, indent=2)

#### Job descriptions (17014 elements) **DONE**

In [37]:
# We first prepare checkpoints (very long computation)
## File to save intermediate results
job_checkpoint = checkpoint_dir / "normalized_jobs.json"

## Load existing checkpoint if available
if job_checkpoint.exists():
    print("Checkpoint found")
    with open(job_checkpoint, "r") as f:
        normalized_jobs = json.load(f)
    processed_ids = {r['job_id'] for r in normalized_jobs}
    print(len(processed_ids), "job postings")
else:
    normalized_jobs = []
    processed_ids = set()

After some tests, the whole extraction is expected to last about : 6h33min, mostly due to normalization through LLM calls.

In [58]:
for job in tqdm(jobs, desc="Normalizing job descriptions"):
    job_id = job['job_id']
    # Skip resumes already processed
    if job_id in processed_ids:
        continue
    job_text = job['text']
    # We normalize the text
    norm_job_des = normalize_job(job_text)# --> None if normalization failed after 3 attempts <--
    if norm_job_des == None:
        processed_ids.add(job_id)
        continue
    # We join everythin into a dictionnary for clarity
    normalized_job = {
        'job_id': job_id,
        'job_description': norm_job_des
    }
    # We put the result in a list
    normalized_jobs.append(normalized_job)
    processed_ids.add(job_id)
    # We save a checkpoint (every 100 job descriptions)
    if len(normalized_jobs) % 100 == 0:
        with open(job_checkpoint, "w") as f:
            json.dump(normalized_jobs, f, indent=2)

# Final save
with open(job_checkpoint, "w") as f:
    json.dump(normalized_jobs, f, indent=2)

Normalizing job descriptions: 100%|██████████| 17014/17014 [3:41:08<00:00,  1.28it/s]  


In [59]:
# Load normalized jobs
with open(job_checkpoint, "r") as f:
    normalized_jobs = json.load(f)
# Let's see how much resumes where not correctly normalized
print("Number of resumes normalized :", len(normalized_jobs), "/", len(jobs))
print("Equivalent in %:", 100*len(normalized_jobs)/len(jobs), "%")

Number of resumes normalized : 17013 / 17014
Equivalent in %: 99.99412248736334 %


 - Number of job descriptions normalized : 17013 / 17014
 - Equivalent in %: 99.99 %

#### Fixing resume normalization **DONE**

Some of the answers from the LLM are not formalized correctly.

Experience is sometimes a list of dictionnaries (which breaks). 
- Ex : [{'Title': '...', 'Company': '...', 'Years': '...', ...}, {'Title': '...', 'Company': '...',...}]

It has to be a list of strings.
- Ex : ['Accountant – Omba Executive Leadership (2016-2018): ...', 'Accountant – Google (2015): ...']

And we have the same problem with the education field.

In [394]:
# Load normalized resumes
with open(resume_checkpoint, "r") as f:
    normalized_resumes = json.load(f)

First, we need to verify all the resumes that were not normalized correctly, and why.

In [ ]:
def validate_resume_record(norm_text:dict):
    """
    Validates that:
    - skills, experience, education, certifications and industries are a list of strings
    """
    # Fields that must be lists of strings
    list_fields = [
        "skills",
        "experience",
        "education",
        "certifications",
        "industries"
    ]

    # verify they are lists
    for field in list_fields:
        if field in norm_text:
            if not isinstance(norm_text[field], list):
                return False
            if not all(isinstance(ele, str) for ele in norm_text[field]):
                return False
    return True

# We search for any errors :
count = 0
for i in range(len(normalized_resumes)):
    job_des = normalized_resumes[i]['norm_text']
    if not validate_resume_record(job_des):
        count += 1
print(count, "ERRORS")

5228 ERRORS


Let's see the error details in each field.

In [ ]:
import agents.fixing_agent
# Reload the file to take into account the changes
importlib.reload(agents.fixing_agent)

In [ ]:
from agents.fixing_agent import detailed_error

**experience**

In [412]:
experience_errors = {
    "no field":0,
    "no error":0,
    "dictionnaries in list":0,
    "lists in list":0,
    "other":0
}

# We count the errors
for i in range(len(normalized_resumes)):
    norm_text = normalized_resumes[i].get('norm_text', {})
    error = detailed_error(norm_text, 'experience')
    experience_errors[error] += 1

# Pretty print
print(" --- SUMMARY --- ")
print("Well normalized experience field (list with strings):")
print(experience_errors['no error'], "/", len(normalized_resumes))
print("No experience field :")
print(experience_errors['no field'], "/", len(normalized_resumes))
print("Badly normalized experience field (list with dictionnaries):")
print(experience_errors['dictionnaries in list'], "/", len(normalized_resumes))
print("Badly normalized experience field (list of lists):")
print(experience_errors['lists in list'], "/", len(normalized_resumes))
print("Badly normalized experience field (other):")
print(experience_errors['other'], "/", len(normalized_resumes))

 --- SUMMARY --- 
Well normalized experience field (list with strings):
8240 / 13314
No experience field :
15 / 13314
Badly normalized experience field (list with dictionnaries):
5048 / 13314
Badly normalized experience field (list of lists):
11 / 13314
Badly normalized experience field (other):
0 / 13314


Lists of errors that make the experience field badly normalized : 
 - Well normalized experience field (list of strings) : 8240 / 13314
 - No experience field : 15 / 13314
 - Badly normalized experience field (list with dictionnaries) : 5048 / 13314
 - Badly normalized experience field (list of lists): 11 / 13314
 - Badly normalized experience field (other): 0 / 13314

**education**

In [414]:
education_errors = {
    "no error":0,
    "no field":0,
    "dictionnaries in list":0,
    "lists in list":0,
    "other":0
}

# We count the errors
for i in range(len(normalized_resumes)):
    norm_text = normalized_resumes[i].get('norm_text', {})
    error = detailed_error(norm_text, 'education')
    education_errors[error] += 1

# Pretty print
print(" --- SUMMARY --- ")
print("Well normalized education field (list with strings):")
print(education_errors['no error'], "/", len(normalized_resumes))
print("No education field :")
print(education_errors['no field'], "/", len(normalized_resumes))
print("Badly normalized education field (list with dictionnaries):")
print(education_errors['dictionnaries in list'], "/", len(normalized_resumes))
print("Badly normalized education field (list of lists):")
print(education_errors['lists in list'], "/", len(normalized_resumes))
print("Badly normalized education field (other):")
print(education_errors['other'], "/", len(normalized_resumes))

 --- SUMMARY --- 
Well normalized education field (list with strings):
7714 / 13314
No education field :
377 / 13314
Badly normalized education field (list with dictionnaries):
5212 / 13314
Badly normalized education field (list of lists):
11 / 13314
Badly normalized education field (other):
0 / 13314


Lists of errors that make the education field badly normalized : 
 - Well normalized education field (list of strings) : 7714 / 13314
 - No education field : 377 / 13314
 - Badly normalized education field (list with dictionnaries) : 5212 / 13314
 - Badly normalized education field (list of lists): 11 / 13314
 - Badly normalized education field (other): 0 / 13314

Then, we need to fix the errors in those fields.

In [ ]:
import agents.fixing_agent
# Reload the file to take into account the changes
importlib.reload(agents.fixing_agent)

<module 'agents.normalization_agent' from 'C:\\Users\\Sebastien\\Desktop\\LLM and GenAI\\Smart_Resume_to_Job_Matcher\\agents\\normalization_agent.py'>

In [ ]:
from agents.fixing_agent import coerce_to_strings_experience, coerce_to_strings_education
from agents.fixing_agent import fix_error_experience, fix_error_education

**experience**

In [418]:
for i in range(len(normalized_resumes)):
    norm_text = normalized_resumes[i].get('norm_text', {})
    normalized_resumes[i]['norm_text'] = fix_error_experience(norm_text)

**education**

In [424]:
for i in range(len(normalized_resumes)):
    norm_text = normalized_resumes[i].get('norm_text', {})
    normalized_resumes[i]['norm_text'] = fix_error_education(norm_text)

And we eventually save the resulting normalized files.

In [428]:
# Saving
with open(resume_checkpoint, "w") as f:
    json.dump(normalized_resumes, f, indent=2)

#### Fixing job normalization **DONE**

We verify if there are any fields which types are not correct.

In [380]:
# Load normalized job descriptions
with open(job_checkpoint, "r") as f:
    normalized_jobs = json.load(f)

In [392]:
def validate_job_record(job_description: dict) -> bool:
    """
    Validates that:
    - job_title, required_experience, required_education, industry are strings
    - required_skills is a list of strings
    """

    # Fields that must be strings
    string_fields = [
        "job_title",
        "required_experience",
        "required_education",
        "industry"
    ]

    for field in string_fields:
        if field in job_description and not isinstance(job_description[field], str):
            return False

    # Validate required_skills
    if "required_skills" in job_description:
        if not isinstance(job_description["required_skills"], list):
            return False
        if not all(isinstance(skill, str) for skill in job_description["required_skills"]):
            return False

    return True

In [393]:
# None don't work, job records are clean
for i in range(len(normalized_jobs)):
    job_des = normalized_jobs[i]['job_description']
    if not validate_job_record(job_des):
        print('ERROR')

This code returns no 'ERROR' message, meaning that every record was handled correctly.

In [ ]:
# Saving (not really necessary as nothing changed)
with open(job_checkpoint, "w") as f:
    json.dump(normalized_jobs, f, indent=2)

# Generate embeddings

### Complete embedding generation

In this step, we will generate the embedding of both resumes and jobs thanks to a prompt. The objects will look like this :
```python 
embedded_resumes = [
    {
    'resume_id': 1, 
    'resume_vector': List[float]
    },
    ...
]

embedded_jobs = [
    {
    'job_id': 1, 
    'job_vector': List[float]
    },
    ...
]
```

In [ ]:
import embeddings.embedding_engine
# Reload the file to take into account the changes
importlib.reload(embeddings.embedding_engine)

In [56]:
from embeddings.embedding_engine import embed_resume_nomic, embed_job_nomic, embed_resume_BGE, embed_job_BGE

#### Resumes (13314 elements) **DONE**

In [57]:
from pathlib import Path
import json
from tqdm import tqdm

In [76]:
def checkpoint_verification_resumes(normalized_resumes_file:Path, embedded_path:Path):
    # Load normalized resumes
    with open(normalized_resumes_file, "r") as f:
        normalized_resumes = json.load(f)
        print("Normalized files imported")

    # Load existing embeddings if checkpoint exists (restart-safe)
    if embedded_path.exists():
        print("Checkpoint found")
        with open(embedded_path, "r", encoding="utf-8") as f:
            embedded = json.load(f)
        processed_ids = {r['resume_id'] for r in embedded}
        print(len(processed_ids), "resume embeddings found")
    else:
        print("No checkpoint found")
        embedded = []
        processed_ids = set()
    
    return normalized_resumes, embedded, processed_ids

**NOMIC embedding model** (~20 minutes)

In [77]:
## First we intitialize our storage (we download it if some was already computed)
# Paths
normalized_resumes_file = Path("checkpoints/normalized_resumes.json")
resume_emb_file = Path("checkpoints/resume_embeddings_nomic.json")
normalized_resumes, embedded_resumes, processed_ids = checkpoint_verification_resumes(normalized_resumes_file, resume_emb_file)

Normalized files imported
Checkpoint found
13296 resume embeddings found


In [ ]:
##  Then generate embeddings for remaining resumes
for r in tqdm(normalized_resumes, desc="Generating resume embeddings..."):
    resume_id = r["resume_id"]
    # Skip resumes already processed
    if resume_id in processed_ids:
        continue
    try:
        resume_vector = embed_resume_nomic(r["norm_text"])
        embedded_resume = {
            'resume_id': resume_id,
            'resume_vector': resume_vector
        }
        embedded_resumes.append(embedded_resume)
        processed_ids.add(resume_id)
    except Exception as e:
        print(f"⚠ Failed embedding for resume {resume_id}: {e}")
    
    # We save a checkpoint (every 100 embedded resumes)
    if len(embedded_resumes) % 100 == 0:
        with open(resume_emb_file, "w", encoding="utf-8") as f:
            json.dump(embedded_resumes, f)

# Save embeddings to disk for later use
with open(resume_emb_file, "w", encoding="utf-8") as f:
    json.dump(embedded_resumes, f)

Generating resume embeddings...:  23%|██▎       | 3113/13314 [01:45<15:36, 10.89it/s]

⚠ Failed embedding for resume 3114: unhashable type: 'dict'


Generating resume embeddings...:  23%|██▎       | 3128/13314 [01:45<07:46, 21.84it/s]

⚠ Failed embedding for resume 3125: unhashable type: 'dict'


Generating resume embeddings...:  27%|██▋       | 3555/13314 [02:05<03:51, 42.21it/s]

...

Generating resume embeddings...:  95%|█████████▍| 12596/13314 [14:12<00:15, 47.76it/s]

⚠ Failed embedding for resume 12653: unhashable type: 'dict'


Generating resume embeddings...:  95%|█████████▍| 12615/13314 [14:12<00:13, 53.44it/s]

⚠ Failed embedding for resume 12670: unhashable type: 'dict'


Generating resume embeddings...: 100%|██████████| 13314/13314 [15:32<00:00, 14.28it/s]


Despite thiniking we took care of the typing issue in the previous phase, apparently, normalization was not strict enough and some files are still rejected from embedding because of typing.

In [446]:
print(f"Generated embeddings: {len(embedded_resumes)} resumes")

print(f"Number of resumes not embedded : {len(normalized_resumes)-len(embedded_resumes)}")
print(f"In % : {100*(len(normalized_resumes)-len(embedded_resumes))/len(normalized_resumes)}")

Generated embeddings: 13296 resumes
Number of resumes not embedded : 18
In % : 0.135196034249662


However, only 18 files were not correctly embedded, which represents less than 0.2% of all resumes.

**BGE embedding model** (~30 minutes)

In [78]:
normalized_resumes_file = Path("checkpoints/normalized_resumes.json")
resume_emb_file = Path("checkpoints/resume_embeddings_bge.json")
normalized_resumes, embedded_resumes, processed_ids = checkpoint_verification_resumes(normalized_resumes_file, resume_emb_file)

Normalized files imported
Checkpoint found
13296 resume embeddings found


In [60]:
##  Then generate embeddings for remaining resumes
for r in tqdm(normalized_resumes, desc="Generating resume embeddings..."):
    resume_id = r["resume_id"]
    # Skip resumes already processed
    if resume_id in processed_ids:
        continue
    try:
        resume_vector = embed_resume_BGE(r["norm_text"])
        embedded_resume = {
            'resume_id': resume_id,
            'resume_vector': resume_vector
        }
        embedded_resumes.append(embedded_resume)
        processed_ids.add(resume_id)
    except Exception as e:
        print(f"⚠ Failed embedding for resume {resume_id}: {e}")
    
    # We save a checkpoint (every 100 embedded resumes)
    if len(embedded_resumes) % 100 == 0:
        with open(resume_emb_file, "w", encoding="utf-8") as f:
            json.dump(embedded_resumes, f)

# Save embeddings to disk for later use
with open(resume_emb_file, "w", encoding="utf-8") as f:
    json.dump(embedded_resumes, f)

Generating resume embeddings...:  23%|██▎       | 3112/13314 [06:31<17:53,  9.51it/s]  

⚠ Failed embedding for resume 3114: unhashable type: 'dict'


Generating resume embeddings...:  23%|██▎       | 3123/13314 [06:32<12:27, 13.63it/s]

⚠ Failed embedding for resume 3125: unhashable type: 'dict'


Generating resume embeddings...:  27%|██▋       | 3548/13314 [07:14<10:51, 14.98it/s]  

...

Generating resume embeddings...:  95%|█████████▍| 12591/13314 [29:11<00:54, 13.36it/s]

⚠ Failed embedding for resume 12653: unhashable type: 'dict'


Generating resume embeddings...:  95%|█████████▍| 12606/13314 [29:12<00:51, 13.80it/s]

⚠ Failed embedding for resume 12670: unhashable type: 'dict'


Generating resume embeddings...: 100%|██████████| 13314/13314 [31:18<00:00,  7.09it/s]


In [79]:
print(f"Generated embeddings: {len(embedded_resumes)} resumes")

print(f"Number of resumes not embedded : {len(normalized_resumes)-len(embedded_resumes)}")
print(f"In % : {100*(len(normalized_resumes)-len(embedded_resumes))/len(normalized_resumes)}")

Generated embeddings: 13296 resumes
Number of resumes not embedded : 18
In % : 0.135196034249662


Here too, only 18 files were not correctly embedded, which represents less than 0.2% of all resumes.

#### Job descriptions (17013 elements) **DONE**

In [62]:
def checkpoint_verification_jobs(normalized_jobs_file:Path, embedded_path:Path):
    # Load normalized resumes
    with open(normalized_jobs_file, "r") as f:
        normalized_jobs = json.load(f)
        print("Normalized files imported")

    # Load existing embeddings if checkpoint exists (restart-safe)
    if embedded_path.exists():
        print("Checkpoint found")
        with open(embedded_path, "r", encoding="utf-8") as f:
            embedded = json.load(f)
        processed_ids = {r['job_id'] for r in embedded}
        print(len(processed_ids), "job embeddings found")
    else:
        print("No checkpoint found")
        embedded = []
        processed_ids = set()
    
    return normalized_jobs, embedded, processed_ids

In [65]:
def embed_job(normalized_jobs, embedded_jobs, processed_ids, job_emb_file, embedding_function):
    for j in tqdm(normalized_jobs, desc="Generating job embeddings..."):
        job_id = j["job_id"]
        # Skip resumes already processed
        if job_id in processed_ids:
            continue
        try:
            job_vector = embedding_function(j["job_description"])
            embedded_job = {
                'job_id': job_id,
                'job_vector': job_vector
            }
            embedded_jobs.append(embedded_job)
            processed_ids.add(job_id)
        except Exception as e:
            print(f"⚠ Failed embedding for job {job_id}: {e}")
        
        # We save a checkpoint (every 100 embedded jobs)
        if len(embedded_jobs) % 100 == 0:
            with open(job_emb_file, "w", encoding="utf-8") as f:
                json.dump(embedded_jobs, f)

    # Save embeddings to disk for later use
    with open(job_emb_file, "w", encoding="utf-8") as f:
        json.dump(embedded_jobs, f)
    
    return embedded_jobs

**NOMIC embedding model** (~23 minutes)

In [71]:
## First we intitialize our storage (we download it if some was already computed)
# Paths to checkpointed normalized data
normalized_jobs_file = Path("checkpoints/normalized_jobs.json")
job_emb_file = Path("checkpoints/jobs_embeddings_nomic.json")
# Checkpoint verification
normalized_jobs, embedded_jobs, processed_ids = checkpoint_verification_jobs(normalized_jobs_file, job_emb_file)
# Embedding generation
embedded_jobs = embed_job(normalized_jobs, embedded_jobs, processed_ids, job_emb_file, embed_job_nomic)

Normalized files imported
Checkpoint found
17013 job embeddings found


Generating job embeddings...: 100%|██████████| 17013/17013 [00:00<00:00, 1652869.78it/s]


In [72]:
print(f"Generated embeddings: {len(embedded_jobs)} job descriptions")

print(f"Number of resumes not embedded : {len(normalized_jobs)-len(embedded_jobs)}")
print(f"In % : {100*(len(normalized_jobs)-len(embedded_jobs))/len(normalized_jobs)}")

Generated embeddings: 17013 job descriptions
Number of resumes not embedded : 0
In % : 0.0


**BGE embedding model** (~30 minutes)

In [73]:
## First we intitialize our storage (we download it if some was already computed)
# Paths to checkpointed normalized data
normalized_jobs_file = Path("checkpoints/normalized_jobs.json")
job_emb_file = Path("checkpoints/jobs_embeddings_bge.json")
# Checkpoint verification
normalized_jobs, embedded_jobs, processed_ids = checkpoint_verification_jobs(normalized_jobs_file, job_emb_file)
# Embedding generation
embedded_jobs = embed_job(normalized_jobs, embedded_jobs, processed_ids, job_emb_file, embed_job_BGE)

Normalized files imported
Checkpoint found
17013 job embeddings found


Generating job embeddings...: 100%|██████████| 17013/17013 [00:00<00:00, 1231494.10it/s]


In [74]:
print(f"Generated embeddings: {len(embedded_jobs)} job descriptions")

print(f"Number of resumes not embedded : {len(normalized_jobs)-len(embedded_jobs)}")
print(f"In % : {100*(len(normalized_jobs)-len(embedded_jobs))/len(normalized_jobs)}")

Generated embeddings: 17013 job descriptions
Number of resumes not embedded : 0
In % : 0.0


However, when it comes to jobs, just like normalization, no job description impeded the process. This is probably because the typing was simpler, no dictionnary inside dictionnaries.

# Similarity and ranking

In [4]:
# Add the parent folder of ingestion to sys.path
project_root = Path("..").resolve()  # notebooks/ is one level down
sys.path.append(str(project_root))

## Format conversion

For now, I have embeddings looking like this : 
```python 
embedded_resumes = [
    {
    'resume_id': 1, 
    'resume_vector': List[float]
    },
    ...
]

embedded_jobs = [
    {
    'job_id': 1, 
    'job_vector': List[float]
    },
    ...
]
```
However, this format takes more space and is harder to manipulate. So, we will first convert our lists into dictionnaries, whose keys will be the embeddings' id.

In [5]:
import embeddings.embedding_format_conversion
# Reload the file to take into account the changes
importlib.reload(embeddings.embedding_format_conversion)

<module 'embeddings.embedding_format_conversion' from 'C:\\Users\\Sebastien\\Desktop\\LLM and GenAI\\Smart_Resume_to_Job_Matcher\\embeddings\\embedding_format_conversion.py'>

In [6]:
from embeddings.embedding_format_conversion import lists_to_id_vector_dicts
import json

In [7]:
## First we import the embeddings
# Paths
resume_emb_file_nomic = Path("checkpoints/resume_embeddings_nomic.json")
resume_emb_file_bge = Path("checkpoints/resume_embeddings_bge.json")
job_emb_file_nomic = Path("checkpoints/jobs_embeddings_nomic.json")
job_emb_file_bge = Path("checkpoints/jobs_embeddings_bge.json")

# Load embedded resumes
with open(resume_emb_file_nomic, "r") as f:
    embedded_resumes_nomic = json.load(f)
    print("Embedded resumes (Nomic) imported")
with open(resume_emb_file_bge, "r") as f:
    embedded_resumes_bge = json.load(f)
    print("Embedded resumes (BGE) imported")

# Load embedded job descriptions
with open(job_emb_file_nomic, "r") as f:
    embedded_jobs_nomic = json.load(f)
    print("Embedded job descriptions (Nomic) imported")
with open(job_emb_file_bge, "r") as f:
    embedded_jobs_bge = json.load(f)
    print("Embedded job descriptions (BGE) imported")

Embedded resumes (Nomic) imported
Embedded resumes (BGE) imported
Embedded job descriptions (Nomic) imported
Embedded job descriptions (BGE) imported


In [8]:
# Then, convert them into smaller dictionnaries for better storage
embedded_resumes_nomic_dict, embedded_jobs_nomic_dict = lists_to_id_vector_dicts(embedded_resumes_nomic, embedded_jobs_nomic)
embedded_resumes_bge_dict, embedded_jobs_bge_dict = lists_to_id_vector_dicts(embedded_resumes_bge, embedded_jobs_bge)

print("Resume IDs (Nomic):", list(embedded_resumes_nomic_dict.keys())[:5])
print("Job IDs (Nomic):", list(embedded_jobs_nomic_dict.keys())[:5])
print("Resume IDs (BGE):", list(embedded_resumes_bge_dict.keys())[:5])
print("Job IDs (BGE):", list(embedded_jobs_bge_dict.keys())[:5])

Resume IDs (Nomic): [1, 2, 3, 4, 5]
Job IDs (Nomic): [1, 2, 3, 4, 5]
Resume IDs (BGE): [1, 2, 3, 4, 5]
Job IDs (BGE): [1, 2, 3, 4, 5]


## Scoring and matching

Now that the embeddings are converted back to dictionnaries, we can use them for scoring and ranking.

In [16]:
import match_engine_and_explanation.match_engine
# Reload the file to take into account the changes
importlib.reload(match_engine_and_explanation.match_engine)

<module 'match_engine_and_explanation.match_engine' from 'C:\\Users\\Sebastien\\Desktop\\LLM and GenAI\\Smart_Resume_to_Job_Matcher\\match_engine_and_explanation\\match_engine.py'>

In [17]:
import ingestion.preprocess
importlib.reload(ingestion.preprocess)

from ingestion.preprocess import resumes_to_raw_text, jobs_to_raw_text

resumes = resumes_to_raw_text("../data/resumes/resumes.csv")
jobs = jobs_to_raw_text("../data/jobs/job_postings.csv")

In [18]:
from match_engine_and_explanation.match_engine import match_jobs_to_resume, match_resumes_to_job, cosine_similarity, eucl_distance

Let's verify that everything works.

**Match resumes to job.**

In [19]:
# Let's remind that :
# embedded_resumes = {resume_id:resume_vector, ...}
# embedded_jobs = {job_id:job_vector, ...}
job_id = 78
# We embed it
job_nomic_embedding = embedded_jobs_nomic_dict[job_id]
job_bge_embedding = embedded_jobs_bge_dict[job_id]
# Choose the similarity
cos_sim = cosine_similarity
eucl_sim = eucl_distance

In [21]:
# Then we try and find the best resume for this job description, depending on the similarity and embedding
## Nomic embedding + Cosine similarity
results_nomic_cos = match_resumes_to_job(
    job_id=job_id,
    job_embedding=job_nomic_embedding,
    resume_embeddings=embedded_resumes_nomic_dict,
    similarity_func=cos_sim,
    top_k=5
)
## Nomic embedding + Euclidian similarity
results_nomic_eucl = match_resumes_to_job(
    job_id=job_id,
    job_embedding=job_nomic_embedding,
    resume_embeddings=embedded_resumes_nomic_dict,
    similarity_func=eucl_sim,
    top_k=5
)
## BGE embedding + Cosine similarity
results_bge_cos = match_resumes_to_job(
    job_id=job_id,
    job_embedding=job_bge_embedding,
    resume_embeddings=embedded_resumes_bge_dict,
    similarity_func=cos_sim,
    top_k=5
)
## BGE embdding + Euclidian similarity
results_bge_eucl = match_resumes_to_job(
    job_id=job_id,
    job_embedding=job_bge_embedding,
    resume_embeddings=embedded_resumes_bge_dict,
    similarity_func=eucl_sim,
    top_k=5
)

Now let's compare the results to see which embedding+similarity combination is the most suitable (non-exhaustive).

In [ ]:
# Job description (Software development)
print(f"Job ID: {job_id}")
print("=" * 30, "Job description", "=" * 30)
print(jobs[job_id-1]['text'])

Job ID: 78
============================== Job description ==============================
title: Mid-Senior .NET or Xamarin Developer. location: US, GA, Atlanta. description: DataFinch Technologies is the leader in electronic data collection for the Autism and Special education fields. We have created a revolutionary, one of a kind, software product targeting: schools, physicians, in-home autism providers and non-profit organizations. Come work for a fast-paced, fun, technology company in a growing field and experience all the latest and greatest cutting edge technologies. Candidates for this role will be assisting in daily programming activities. The company has been completely bootstrapped organically and we currently have no investors. We have over 15,000 licensed users and this year we broke 2 million in revenue. We are located near perimeter mall.ResponsibilitiesAs a developer at DataFinch you will be a key contributor delivering new product features and functionality. Using an agi

In [24]:
# We only look at the first results ofr each combination
best_nomic_cos = results_nomic_cos[0]
best_nomic_eucl = results_nomic_eucl[0]
best_bge_cos = results_bge_cos[0]
best_bge_eucl = results_bge_eucl[0]
for r in [best_nomic_cos, best_nomic_eucl, best_bge_cos, best_bge_eucl]:
    print("=" * 80)
    resume_id = r['resume_id']
    print(f"Resume ID: {resume_id}")
    print(f"Resume's text : ")
    print(resumes[resume_id-1]['text'])
    print(f"Similarity score: {r['score']:.3f}")

Resume ID: 4095
Resume's text : 
Category: Digital Media. Resume text: digital media entrylevel resume objective create statement describing naming position seeking obtain may also include specific field area interested skills expertise could contribute company section optional could omitted substituted career highlights summary qualifications section education institution city state write full name expected degree also list minors concentrations gpa 4040 recommended include gpa 30 higher expected graduation month year possible information include study abroad experiences relevant coursework academic awards honors technical skills utilities languages software databases hardware experience relevant experience internship experience possible titlesdescriptive headings organization job titlecity state month yearmonth year position header one example formatting formats include using two lines rearranging order organization job title location dates emphasize important elements bold italicize

For the matching of resumes to a job, we will now be using the combination of **BGE embedding with cosine similarity**, as it seems to be the most appropirate in for this example.

**Match jobs to resume.**

In [39]:
# Let's remind that :
# embedded_resumes = {resume_id:resume_vector, ...}
# embedded_jobs = {job_id:job_vector, ...}
resume_id = 846
# We embed it
resume_nomic_embedding = embedded_resumes_nomic_dict[resume_id]
resume_bge_embedding = embedded_resumes_bge_dict[resume_id]
# Choose the similarity
cos_sim = cosine_similarity
eucl_sim = eucl_distance

In [40]:
# Then we try and find the best resume for this job description, depending on the similarity and embedding
## Nomic embedding + Cosine similarity
results_nomic_cos = match_jobs_to_resume(
    resume_id=resume_id,
    resume_embedding=resume_nomic_embedding,
    job_embeddings=embedded_jobs_nomic_dict,
    similarity_func=cos_sim,
    top_k=5
)
## Nomic embedding + Euclidian similarity
results_nomic_eucl = match_jobs_to_resume(
    resume_id=resume_id,
    resume_embedding=resume_nomic_embedding,
    job_embeddings=embedded_jobs_nomic_dict,
    similarity_func=eucl_sim,
    top_k=5
)
## BGE embedding + Cosine similarity
results_bge_cos = match_jobs_to_resume(
    resume_id=resume_id,
    resume_embedding=resume_bge_embedding,
    job_embeddings=embedded_jobs_bge_dict,
    similarity_func=cos_sim,
    top_k=5
)
## BGE embdding + Euclidian similarity
results_bge_eucl = match_jobs_to_resume(
    resume_id=resume_id,
    resume_embedding=resume_bge_embedding,
    job_embeddings=embedded_jobs_bge_dict,
    similarity_func=eucl_sim,
    top_k=5
)

In [ ]:
# Resume (A consultant in communication)
print(f"Resume ID: {resume_id}")
print("=" * 30, "Resume text", "=" * 30)
print(resumes[resume_id-1]['text'])

Resume ID: 846
============================== Resume text ==============================
Category: Consultant. Resume text: norris lebsack 56944 renaldo camp los angeles ca phone 1 555 190 4102 experience lebsack rosenbaum roob chicago il consultant 052020 present experience visio publisher perform issue identification communication resolution issues ranging moderate complex ensure clients stay informed new features available monthly release calls communicate concise consistent status updates customer assist identifying usage gaps increase adoption levels successful experience working directly customers resolve complex implementation issues block inc chicago il consultant 112017 122019 openminded capable establishing excellent cooperation business partners internalexternal experience knowledge internal audit assurance functions internal controls compliance enterprise risk management excellent presentation skills demonstrating flair enthusiasm practical administration skills linux unix 

In [42]:
# We only look at the first results ofr each combination
best_nomic_cos = results_nomic_cos[0]
best_nomic_eucl = results_nomic_eucl[0]
best_bge_cos = results_bge_cos[0]
best_bge_eucl = results_bge_eucl[0]
for j in [best_nomic_cos, best_nomic_eucl, best_bge_cos, best_bge_eucl]:
    print("=" * 80)
    job_id = j['job_id']
    print(f"Job ID: {job_id}")
    print(f"Job's description : ")
    print(jobs[job_id-1]['text'])
    print(f"Similarity score: {j['score']:.3f}")

Job ID: 16942
Job's description : 
title: Sales Consultant NYC. location: US, NY, New York. description: EXPERIENCED SALES PROFESSIONALS ONLY!EARN $50,000 to $200,000 PLUS BENEFITS A rare opportunity for a sales professional to upset a whole industry with a first of its kind software product that is revolutionizing the way the restaurant industry does business. Only apply if you can walk into any restaurant, find the decision maker and close the sale. Sarbari is hiring super motivated, highly skilled sales professionals. Your primary job will be getting in front of key decision makers for restaurants, caterers, and institutions with food service to present an online, subscription-based software solution that is guaranteed to save a lot of money and time. Candidates must have proven cold calling and closing skills. Existing contacts in the industry are a huge bonus.Sarbari offers a base salary and high earning potential including a complete benefits package. We are a fast paced, high-en

They all seem to kind of match the resume, but, for the matching of jobs to a resume, we will now be using the combination of **BGE embedding with cosine similarity**, as it was the most appropriate for the last part.

# LLM-based explanation

In [50]:
import match_engine_and_explanation.llm_explanation
# Reload the file to take into account the changes
importlib.reload(match_engine_and_explanation.llm_explanation)

<module 'match_engine_and_explanation.llm_explanation' from 'C:\\Users\\Sebastien\\Desktop\\LLM and GenAI\\Smart_Resume_to_Job_Matcher\\match_engine_and_explanation\\llm_explanation.py'>

In [52]:
from match_engine_and_explanation.llm_explanation import generate_match_explanation

We generate the matching explanation for the 2 best matching of the last part.

In [59]:
# We need to import the normalized jobs and resumes
# Paths
normalized_resumes = Path("checkpoints/normalized_resumes.json")
normalized_jobs = Path("checkpoints/normalized_jobs.json")

# Load resumes
with open(normalized_resumes, "r") as f:
    norm_resumes = json.load(f)
    print("Normalized resumes imported")

# Load job descriptions
with open(normalized_jobs, "r") as f:
    norm_jobs = json.load(f)
    print("Normalized job descriptions imported")

Normalized resumes imported
Normalized job descriptions imported


**Best match**

In [61]:
# We get all the information we need from the best match
r_0 = results_bge_cos[0]
job_id = r_0['job_id']
resume_id = r_0['resume_id']
score = r_0['score']

In [71]:
# Then we can get the description of each resume and job
norm_resume = None
for d in norm_resumes:
    if d.get('resume_id') == resume_id:
        norm_resume = d['norm_text']
        break
    
norm_job = None
for d in norm_jobs:
    if d.get('job_id') == job_id:
        norm_job = d['job_description']
        break

In [80]:
explanation = generate_match_explanation(norm_job, norm_resume, score)

In [81]:
print(explanation)

Based on the job description and candidate profile, I assess the similarity score as follows:

**Key Strengths:**

1. The candidate has experience in consulting roles, which aligns with the job's requirement for end-to-end implementation experience.
2. They have skills in Microsoft Office, Visio, and Publisher, which are relevant to the job's mention of UI Map and Business script.

**Gaps or Risks:**

1. The candidate lacks direct experience with Cc&B (Concurrent Technologies Corporation & Beyond) products, which is a critical requirement for the job.
2. Their education background is in Business Skills, whereas the job requires technical expertise in Information Technology and Services.
3. The candidate's certifications are not specified, which may be a concern for a role that demands specific technical skills.

**Overall Suitability:**

Given the similarity score of 0.718, I would say that the candidate has some relevant experience and skills, but there are significant gaps in their b

**Second best match**

In [82]:
# We get all the information we need from the best match
r_1 = results_bge_cos[1]
job_id = r_1['job_id']
resume_id = r_1['resume_id']
score = r_1['score']

In [83]:
# Then we can get the description of each resume and job
norm_resume = None
for d in norm_resumes:
    if d.get('resume_id') == resume_id:
        norm_resume = d['norm_text']
        break
    
norm_job = None
for d in norm_jobs:
    if d.get('job_id') == job_id:
        norm_job = d['job_description']
        break

In [84]:
explanation_2 = generate_match_explanation(norm_job, norm_resume, score)

In [85]:
print(explanation_2)

Based on the provided job description and candidate profile, I'll assess the similarity score of 0.714.

**Key Strengths:**

1. The candidate has relevant experience in consulting roles, which demonstrates their ability to work with clients and resolve issues.
2. Their education background in Business Skills from Lim College is a good match for the legal services industry.
3. Although the skills listed don't directly align with the job requirements, the candidate's experience working with customers and resolving complex implementation issues shows potential for adapting to new situations.

**Gaps or Risks:**

1. The candidate lacks direct experience in leadership, communication, advocacy, client account management, and processing client fees, which are essential skills for the job.
2. Their skillset is heavily focused on technical tools like Microsoft Office, Visio, Publisher, Linux, and Unix, whereas the job requires more soft skills and industry-specific knowledge.

**Overall Suitabi

In [1]:
!pip install rank_bm25

In [5]:
import pandas as pd
from rank_bm25 import BM25Okapi

# 1. Load the data 
jobs_df = pd.read_csv('../data/jobs/job_postings.csv')
resumes_df = pd.read_csv('../data/resumes/resumes.csv')

# 2. Prepare the jobs text for the word filter (using 'description')
job_texts = jobs_df['description'].astype(str).tolist() 
tokenized_jobs = [text.lower().split() for text in job_texts]

# 3. Create the word filter (BM25)
bm25 = BM25Okapi(tokenized_jobs)

# 4. Pick the first resume to test (using 'Text')
test_resume = str(resumes_df['Text'].iloc[0]).lower().split()

# 5. Score all jobs against this resume
scores = bm25.get_scores(test_resume)
jobs_df['keyword_match_score'] = scores

# 6. Show the top 5 matching jobs (using 'title')
top_jobs = jobs_df.sort_values(by='keyword_match_score', ascending=False).head(5)
print(top_jobs[['title', 'keyword_match_score']])

                                                   title  keyword_match_score
6136                           Elementary School Teacher            82.772627
6905        Short-Term Leave High School Science Teacher            78.254580
4592                             Hospitalist jobs in USA            77.927803
10266  High School Special Education Building Substit...            75.357880
3926                            English Teacher in China            74.256595


In [7]:
!pip install sentence-transformers scikit-learn

   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   ------------- -------------------------- 262.1/760.7 kB ? eta -:--:--
   ------------- -------------------------- 262.1/760.7 kB ? eta -:--:--
   -------------------------- ----------- 524.3/760.7 kB 932.9 kB/s eta 0:00:01
   -------------------------------------- 760.7/760.7 kB 887.1 kB/s eta 0:00:00
   ---------------------------------------- 0.0/855.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/855.2 kB ? eta -:--:--
   ------------------------ --------------- 524.3/855.2 kB 1.4 MB/s eta 0:00:01
   ------------------------------------ --- 786.4/855.2 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 855.2/855.2 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.9 MB ? eta -:--:--
   ------- ---------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpcore 1.0.2 requires h11<0.15,>=0.13, but you have h11 0.16.0 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load a fast, standard sentence embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Select the first 1000 jobs to keep the test fast
sample_jobs = jobs_df.head(1000).copy()

# 3. Convert the jobs and the resume into semantic embeddings
job_embeddings = model.encode(sample_jobs['description'].astype(str).tolist())
resume_embedding = model.encode([str(resumes_df['Text'].iloc[0])])

# 4. Calculate how similar the meanings are (scores from 0 to 1)
similarities = cosine_similarity(resume_embedding, job_embeddings)[0]
sample_jobs['semantic_score'] = similarities

# 5. Show the top 5 semantic matches
top_semantic_jobs = sample_jobs.sort_values(by='semantic_score', ascending=False).head(5)
print(top_semantic_jobs[['title', 'semantic_score']])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\shree\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shree\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

                                                 title  semantic_score
167                        Customer Service Team Lead         0.461709
93                           Administrative Assistant         0.444746
612  Assistant Training and Quality Development Fac...        0.432760
621                 THIS YEAR: Art Teacher (Part Time)        0.430310
5                                     Accounting Clerk        0.430239


In [11]:
!pip install lightgbm

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------------------------------ --------- 1.0/1.4 MB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 3.2 MB/s eta 0:00:00


In [13]:
import lightgbm as lgb
import numpy as np

# 1. Create a fake "hired" scorecard for training (1 = good match, 0 = bad match)
# We pretend jobs that scored well on both filters are the ones the user actually applied to.
sample_jobs['mock_relevance'] = np.where(
    (sample_jobs['keyword_match_score'] > 40) & (sample_jobs['semantic_score'] > 0.4), 1, 0
)

# 2. Prepare the scores for the judge to learn from
features = sample_jobs[['keyword_match_score', 'semantic_score']]
target = sample_jobs['mock_relevance']

# 3. Create the LightGBM Ranker (This is the LTR model the job description asked for)
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg"
)

# 4. Train the judge! (We tell it all 1000 jobs belong to our 1 test resume)
group = [len(sample_jobs)]
ranker.fit(features, target, group=group)

# 5. Let the trained judge re-rank all our jobs based on what it learned
sample_jobs['final_ltr_score'] = ranker.predict(features)
final_top_jobs = sample_jobs.sort_values(by='final_ltr_score', ascending=False).head(5)

print(final_top_jobs[['title', 'keyword_match_score', 'semantic_score', 'final_ltr_score']])

C:\Users\shree\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\shree\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\shree\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\shree\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\shree\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000165 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 507
[LightGBM] [Info] Number of data points in the train set: 1000, number of used features: 2
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 